In [ ]:
import os, json, math, time
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import matplotlib.pyplot as plt


ROOT_DIR = os.getcwd() 
ART_DIR = os.path.join(ROOT_DIR, "artifacts")
FIG_DIR = os.path.join(ART_DIR, "figures")
os.makedirs(ART_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)


SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [6]:
DATASET_NAME = "CIFAR10"

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_full = datasets.CIFAR10(root="./data", train=True,
                              download=True, transform=transform)
test_ds    = datasets.CIFAR10(root="./data", train=False,
                              download=True, transform=transform)

val_ratio = 0.2
n_total = len(train_full)
n_val = int(n_total * val_ratio)
n_train = n_total - n_val

g = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(train_full, [n_train, n_val], generator=g)

batch_size = 128
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=(device.type=="cuda"))
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=(device.type=="cuda"))
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=(device.type=="cuda"))

xb, yb = next(iter(train_loader))
print("Batch x shape:", xb.shape, "dtype:", xb.dtype)
print("Batch y shape:", yb.shape, "dtype:", yb.dtype)
print("x min/max:", xb.min().item(), xb.max().item())
print("y unique (first batch):", torch.unique(yb)[:10], "...")

100.0%
g:\AICourseMirea\aie-demidov\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Batch x shape: torch.Size([128, 3, 32, 32]) dtype: torch.float32
Batch y shape: torch.Size([128]) dtype: torch.int64
x min/max: 0.0 1.0
y unique (first batch): tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]) ...


In [7]:
class MLP(nn.Module):
    def __init__(
        self,
        input_shape: Tuple[int, int, int],  
        num_classes: int = 10,
        hidden_sizes: List[int] = [512, 256, 128],
        activation: str = "relu",
        dropout_p: float = 0.0,
        use_batchnorm: bool = False,
    ):
        super().__init__()
        c, h, w = input_shape
        in_features = c * h * w

        act_layer = {
            "relu": nn.ReLU,
            "gelu": nn.GELU,
            "tanh": nn.Tanh
        }.get(activation.lower(), nn.ReLU)

        layers = []
        layers.append(nn.Flatten())

        prev = in_features
        for hs in hidden_sizes:
            layers.append(nn.Linear(prev, hs))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(hs))
            layers.append(act_layer())
            if dropout_p and dropout_p > 0:
                layers.append(nn.Dropout(p=dropout_p))
            prev = hs

        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


sample_x, sample_y = train_full[0]
input_shape = tuple(sample_x.shape)
print("Input shape:", input_shape)

Input shape: (3, 32, 32)


In [8]:
def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion) -> Dict[str, float]:
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), y)
        n_batches += 1

    return {
        "loss": total_loss / max(n_batches, 1),
        "acc": total_acc / max(n_batches, 1),
    }

@torch.no_grad()
def evaluate(model, loader, criterion) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1

    return {
        "loss": total_loss / max(n_batches, 1),
        "acc": total_acc / max(n_batches, 1),
    }

In [9]:
class EarlyStopping:
    def __init__(self, patience: int = 4, min_delta: float = 0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best = None
        self.num_bad = 0

    def step(self, metric: float) -> bool:
        """
        Returns True if should stop.
        """
        if self.best is None or metric > self.best + self.min_delta:
            self.best = metric
            self.num_bad = 0
            return False
        self.num_bad += 1
        return self.num_bad >= self.patience


def fit_model(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    epochs: int,
    early_stopping: Optional[EarlyStopping] = None,
) -> Dict[str, List[float]]:
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    for epoch in range(1, epochs + 1):
        tr = train_one_epoch(model, train_loader, optimizer, criterion)
        va = evaluate(model, val_loader, criterion)

        history["train_loss"].append(tr["loss"])
        history["train_acc"].append(tr["acc"])
        history["val_loss"].append(va["loss"])
        history["val_acc"].append(va["acc"])

        if va["acc"] > best_val_acc:
            best_val_acc = va["acc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch

        print(f"Epoch {epoch:02d}/{epochs} | "
              f"train loss {tr['loss']:.4f} acc {tr['acc']:.4f} | "
              f"val loss {va['loss']:.4f} acc {va['acc']:.4f}")

        if early_stopping is not None:
            if early_stopping.step(va["acc"]):
                print(f"EarlyStopping triggered at epoch {epoch}. Best epoch was {best_epoch} (val_acc={best_val_acc:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    history["epochs_trained"] = len(history["train_loss"])
    history["best_val_acc"] = max(history["val_acc"]) if history["val_acc"] else float("nan")
    history["best_val_loss"] = min(history["val_loss"]) if history["val_loss"] else float("nan")
    return history


def plot_curves(history: Dict[str, List[float]], title: str, outpath: str):
    epochs = np.arange(1, len(history["train_loss"]) + 1)

    plt.figure()
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_loss"], label="val_loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

In [10]:
import csv

RUNS_CSV = os.path.join(ART_DIR, "runs.csv")

RUNS_FIELDS = [
    "experiment_id",
    "dataset",
    "seed",
    "model_summary",
    "optimizer",
    "lr",
    "momentum",
    "weight_decay",
    "epochs_trained",
    "best_val_accuracy",
    "best_val_loss",
]

def ensure_runs_csv_header():
    if not os.path.exists(RUNS_CSV):
        with open(RUNS_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=RUNS_FIELDS)
            writer.writeheader()

def append_run(row: Dict[str, Any]):
    ensure_runs_csv_header()
    for k in RUNS_FIELDS:
        if k not in row:
            row[k] = ""
    with open(RUNS_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=RUNS_FIELDS)
        writer.writerow(row)

In [11]:
def run_experiment(
    experiment_id: str,
    model_cfg: Dict[str, Any],
    opt_cfg: Dict[str, Any],
    epochs: int,
    early_stop_patience: Optional[int] = None,
    plot_path: Optional[str] = None,
) -> Dict[str, Any]:
    model = MLP(input_shape=input_shape, num_classes=10, **model_cfg).to(device)
    criterion = nn.CrossEntropyLoss()

    opt_name = opt_cfg["name"].lower()
    lr = opt_cfg.get("lr", 1e-3)
    weight_decay = opt_cfg.get("weight_decay", 0.0)
    momentum = opt_cfg.get("momentum", 0.0)

    if opt_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        opt_label = "Adam"
        momentum_for_log = 0.0
    elif opt_name == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
        opt_label = "SGD"
        momentum_for_log = momentum
    else:
        raise ValueError("Unknown optimizer name. Use 'adam' or 'sgd'.")

    es = EarlyStopping(patience=early_stop_patience) if early_stop_patience is not None else None

    print("\n" + "="*80)
    print(f"Running {experiment_id}")
    print("Model cfg:", model_cfg)
    print("Opt cfg:", opt_cfg, "epochs:", epochs, "early_stop_patience:", early_stop_patience)
    print("="*80)

    history = fit_model(model, optimizer, criterion, epochs=epochs, early_stopping=es)

    if plot_path is not None:
        plot_curves(history, title=experiment_id, outpath=plot_path)

    model_summary = f"hidden={model_cfg.get('hidden_sizes')} act={model_cfg.get('activation','relu')} " \
                    f"dropout={model_cfg.get('dropout_p',0.0)} bn={model_cfg.get('use_batchnorm',False)}"

    row = {
        "experiment_id": experiment_id,
        "dataset": DATASET_NAME,
        "seed": SEED,
        "model_summary": model_summary,
        "optimizer": opt_label,
        "lr": lr,
        "momentum": momentum_for_log if opt_label == "SGD" else 0.0,
        "weight_decay": weight_decay,
        "epochs_trained": history["epochs_trained"],
        "best_val_accuracy": history["best_val_acc"],
        "best_val_loss": history["best_val_loss"],
    }
    append_run(row)

    return {
        "experiment_id": experiment_id,
        "model_cfg": model_cfg,
        "opt_cfg": opt_cfg,
        "history": history,
        "model_state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
        "best_val_acc": history["best_val_acc"],
        "best_val_loss": history["best_val_loss"],
    }



BASE_MODEL_CFG = dict(
    hidden_sizes=[512, 256, 128],
    activation="relu",
    dropout_p=0.0,
    use_batchnorm=False,
)

BASE_OPT_CFG = dict(
    name="adam",
    lr=1e-3,
    weight_decay=0.0,
)

EPOCHS_REG = 15  

res_E1 = run_experiment(
    "E1",
    model_cfg={**BASE_MODEL_CFG, "dropout_p": 0.0, "use_batchnorm": False},
    opt_cfg=BASE_OPT_CFG,
    epochs=EPOCHS_REG,
)

res_E2 = run_experiment(
    "E2",
    model_cfg={**BASE_MODEL_CFG, "dropout_p": 0.3, "use_batchnorm": False},
    opt_cfg=BASE_OPT_CFG,
    epochs=EPOCHS_REG,
)

res_E3 = run_experiment(
    "E3",
    model_cfg={**BASE_MODEL_CFG, "dropout_p": 0.0, "use_batchnorm": True},
    opt_cfg=BASE_OPT_CFG,
    epochs=EPOCHS_REG,
)

best_prev = max([res_E2, res_E3], key=lambda r: r["best_val_acc"])
print("\nBest among E2/E3:", best_prev["experiment_id"], "val_acc=", best_prev["best_val_acc"])


Running E1
Model cfg: {'hidden_sizes': [512, 256, 128], 'activation': 'relu', 'dropout_p': 0.0, 'use_batchnorm': False}
Opt cfg: {'name': 'adam', 'lr': 0.001, 'weight_decay': 0.0} epochs: 15 early_stop_patience: None
Epoch 01/15 | train loss 1.9078 acc 0.2985 | val loss 1.8173 acc 0.3421
Epoch 02/15 | train loss 1.7209 acc 0.3751 | val loss 1.6695 acc 0.3937
Epoch 03/15 | train loss 1.6391 acc 0.4125 | val loss 1.6082 acc 0.4175
Epoch 04/15 | train loss 1.5718 acc 0.4375 | val loss 1.5613 acc 0.4389
Epoch 05/15 | train loss 1.5282 acc 0.4513 | val loss 1.5509 acc 0.4402
Epoch 06/15 | train loss 1.4952 acc 0.4645 | val loss 1.5075 acc 0.4581
Epoch 07/15 | train loss 1.4447 acc 0.4806 | val loss 1.4998 acc 0.4647
Epoch 08/15 | train loss 1.4302 acc 0.4900 | val loss 1.4579 acc 0.4756
Epoch 09/15 | train loss 1.4000 acc 0.4994 | val loss 1.4879 acc 0.4748
Epoch 10/15 | train loss 1.3748 acc 0.5095 | val loss 1.4569 acc 0.4790
Epoch 11/15 | train loss 1.3493 acc 0.5147 | val loss 1.4149 a

In [12]:

BEST_MODEL_CFG = best_prev["model_cfg"]
BEST_OPT_CFG = {**BASE_OPT_CFG} 
EPOCHS_E4 = 30  

best_plot_path = os.path.join(FIG_DIR, "curves_best.png")

res_E4 = run_experiment(
    "E4",
    model_cfg=BEST_MODEL_CFG,
    opt_cfg=BEST_OPT_CFG,
    epochs=EPOCHS_E4,
    early_stop_patience=4,  
    plot_path=best_plot_path,
)

best_model_path = os.path.join(ART_DIR, "best_model.pt")
torch.save(res_E4["model_state_dict"], best_model_path)
print("Saved:", best_model_path)

best_config = {
    "dataset": DATASET_NAME,
    "seed": SEED,
    "experiment_id": "E4",
    "model_cfg": BEST_MODEL_CFG,
    "opt_cfg": BEST_OPT_CFG,
}
best_config_path = os.path.join(ART_DIR, "best_config.json")
with open(best_config_path, "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)
print("Saved:", best_config_path)

print("Saved plot:", best_plot_path)


Running E4
Model cfg: {'hidden_sizes': [512, 256, 128], 'activation': 'relu', 'dropout_p': 0.0, 'use_batchnorm': True}
Opt cfg: {'name': 'adam', 'lr': 0.001, 'weight_decay': 0.0} epochs: 30 early_stop_patience: 4
Epoch 01/30 | train loss 1.6368 acc 0.4130 | val loss 1.7702 acc 0.3617
Epoch 02/30 | train loss 1.4071 acc 0.4958 | val loss 1.5081 acc 0.4685
Epoch 03/30 | train loss 1.2888 acc 0.5398 | val loss 1.4518 acc 0.4880
Epoch 04/30 | train loss 1.1941 acc 0.5748 | val loss 1.4270 acc 0.4972
Epoch 05/30 | train loss 1.1156 acc 0.6016 | val loss 1.3268 acc 0.5336
Epoch 06/30 | train loss 1.0319 acc 0.6335 | val loss 1.5760 acc 0.4659
Epoch 07/30 | train loss 0.9596 acc 0.6581 | val loss 1.6141 acc 0.4809
Epoch 08/30 | train loss 0.8889 acc 0.6857 | val loss 1.5897 acc 0.4785
Epoch 09/30 | train loss 0.8170 acc 0.7108 | val loss 1.6814 acc 0.4915
EarlyStopping triggered at epoch 9. Best epoch was 5 (val_acc=0.5336)
Saved: g:\AICourseMirea\aie-demidov\homeworks\HW08-09\artifacts\best

In [13]:

model_best = MLP(input_shape=input_shape, num_classes=10, **BEST_MODEL_CFG).to(device)
state = torch.load(best_model_path, map_location="cpu")
model_best.load_state_dict(state)

criterion = nn.CrossEntropyLoss()
test_metrics = evaluate(model_best, test_loader, criterion)
print("FINAL TEST (E4) metrics:", test_metrics)

FINAL TEST (E4) metrics: {'loss': 1.3333007625386686, 'acc': 0.53125}


In [14]:

res_O1 = run_experiment(
    "O1",
    model_cfg=BEST_MODEL_CFG,
    opt_cfg={"name": "adam", "lr": 1e-1, "weight_decay": 0.0},
    epochs=8,
)

res_O2 = run_experiment(
    "O2",
    model_cfg=BEST_MODEL_CFG,
    opt_cfg={"name": "adam", "lr": 1e-5, "weight_decay": 0.0},
    epochs=8,
)

res_O3 = run_experiment(
    "O3",
    model_cfg=BEST_MODEL_CFG,
    opt_cfg={"name": "sgd", "lr": 1e-2, "momentum": 0.9, "weight_decay": 1e-4},
    epochs=12,
)

out_lr_plot = os.path.join(FIG_DIR, "curves_lr_extremes.png")

epochs1 = np.arange(1, len(res_O1["history"]["train_loss"]) + 1)
epochs2 = np.arange(1, len(res_O2["history"]["train_loss"]) + 1)

plt.figure()
plt.plot(epochs1, res_O1["history"]["train_loss"], label="O1 train_loss (lr=1e-1)")
plt.plot(epochs1, res_O1["history"]["val_loss"],   label="O1 val_loss (lr=1e-1)")
plt.plot(epochs2, res_O2["history"]["train_loss"], label="O2 train_loss (lr=1e-5)")
plt.plot(epochs2, res_O2["history"]["val_loss"],   label="O2 val_loss (lr=1e-5)")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("LR extremes: too big vs too small")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_lr_plot, dpi=150)
plt.close()

print("Saved plot:", out_lr_plot)
print("runs.csv at:", RUNS_CSV)


Running O1
Model cfg: {'hidden_sizes': [512, 256, 128], 'activation': 'relu', 'dropout_p': 0.0, 'use_batchnorm': True}
Opt cfg: {'name': 'adam', 'lr': 0.1, 'weight_decay': 0.0} epochs: 8 early_stop_patience: None
Epoch 01/8 | train loss 1.8427 acc 0.3391 | val loss 1.7307 acc 0.3789
Epoch 02/8 | train loss 1.6267 acc 0.4143 | val loss 1.6362 acc 0.4126
Epoch 03/8 | train loss 1.5288 acc 0.4518 | val loss 1.7829 acc 0.3733
Epoch 04/8 | train loss 1.4553 acc 0.4810 | val loss 1.6744 acc 0.4156
Epoch 05/8 | train loss 1.3983 acc 0.5042 | val loss 1.6196 acc 0.4461
Epoch 06/8 | train loss 1.3489 acc 0.5203 | val loss 1.6679 acc 0.4281
Epoch 07/8 | train loss 1.3219 acc 0.5332 | val loss 1.5561 acc 0.4565
Epoch 08/8 | train loss 1.2740 acc 0.5455 | val loss 1.4924 acc 0.4694

Running O2
Model cfg: {'hidden_sizes': [512, 256, 128], 'activation': 'relu', 'dropout_p': 0.0, 'use_batchnorm': True}
Opt cfg: {'name': 'adam', 'lr': 1e-05, 'weight_decay': 0.0} epochs: 8 early_stop_patience: None
Ep

In [15]:
import pandas as pd

REPORT_PATH = os.path.join(ROOT_DIR, "report.md")
runs = pd.read_csv(RUNS_CSV)

lines = []

lines.append("# HW08-09 report\n")
lines.append("## Dataset\n")
lines.append(f"- Name: {DATASET_NAME}\n")
lines.append(f"- Seed: {SEED}\n")

lines.append("## Regularization experiments (E1–E4)\n")
for eid in ["E1", "E2", "E3", "E4"]:
    row = runs[runs["experiment_id"] == eid].iloc[0]
    lines.append(
        f"- {eid}: model={row['model_summary']}, "
        f"optimizer={row['optimizer']} (lr={row['lr']}), "
        f"best_val_acc={row['best_val_accuracy']:.4f}, "
        f"best_val_loss={row['best_val_loss']:.4f}\n"
    )

lines.append("\n## LR and optimizers (O1–O3)\n")
for eid in ["O1", "O2", "O3"]:
    row = runs[runs["experiment_id"] == eid].iloc[0]
    lines.append(
        f"- {eid}: optimizer={row['optimizer']} (lr={row['lr']}, "
        f"momentum={row['momentum']}, weight_decay={row['weight_decay']}), "
        f"best_val_acc={row['best_val_accuracy']:.4f}\n"
    )

lines.append("\n## Figures\n")
lines.append("- `artifacts/figures/curves_best.png`\n")
lines.append("- `artifacts/figures/curves_lr_extremes.png`\n")

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.writelines(lines)

print("Saved report to", REPORT_PATH)


Saved report to g:\AICourseMirea\aie-demidov\homeworks\HW08-09\report.md
